In [7]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
from sklearn.model_selection import train_test_split


tf.random.set_seed(42)
np.random.seed(42)


# Define stronger CNN Feature Extractor

def build_cnn_model(input_shape=(28, 28, 1), num_classes=10):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu', name='feature_dense')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs)


cnn_model = build_cnn_model()
print('CNN model built successfully.')


# Load Dataset (MNIST)

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = np.expand_dims(x_train.astype('float32') / 255.0, -1)
x_test = np.expand_dims(x_test.astype('float32') / 255.0, -1)


# Use a larger subset for better feature quality while keeping runtime manageable

cnn_train_size = 15000
feature_train_size = 8000
feature_test_size = 2000

x_cnn = x_train[:cnn_train_size]
y_cnn = y_train[:cnn_train_size]


cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
]

cnn_model.fit(
    x_cnn,
    y_cnn,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    callbacks=cnn_callbacks,
    verbose=1
)


# Create feature extractor

feature_extractor = models.Model(
    inputs=cnn_model.input,
    outputs=cnn_model.get_layer('feature_dense').output
)
print('Feature extractor created successfully!')


# Extract features

train_features = feature_extractor.predict(x_train[:feature_train_size], batch_size=256)
test_features = feature_extractor.predict(x_test[:feature_test_size], batch_size=256)

y_train_small = y_train[:feature_train_size]
y_test_small = y_test[:feature_test_size]

print(f'Features extracted: Train={train_features.shape}, Test={test_features.shape}')


# Define KAN-like Layer and Model

class KANLayer(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.units = units

    def build(self, input_shape):
        self.W = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='glorot_uniform',
            trainable=True
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
        self.alpha = self.add_weight(
            shape=(self.units,),
            initializer='ones',
            trainable=True
        )

    def call(self, inputs):
        linear = tf.matmul(inputs, self.W) + self.b
        nonlinear = tf.math.sin(self.alpha * linear)
        return linear + nonlinear


def build_kan_model(input_dim, num_classes):
    inputs = layers.Input(shape=(input_dim,))
    x = KANLayer(96)(inputs)
    x = layers.Dense(96, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes)(x)  # logits
    return models.Model(inputs, outputs)


kan_model = build_kan_model(train_features.shape[1], 10)
kan_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=8e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
kan_model.summary()


# Train KAN on extracted CNN features

X_train_kan, X_cal, y_train_kan, y_cal = train_test_split(
    train_features,
    y_train_small,
    test_size=0.2,
    random_state=42,
    stratify=y_train_small
)

kan_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
]

kan_model.fit(
    X_train_kan,
    y_train_kan,
    epochs=12,
    batch_size=128,
    validation_data=(X_cal, y_cal),
    callbacks=kan_callbacks,
    verbose=1
)


# Temperature scaling

def learn_temperature(model, x_cal, y_cal, steps=250, lr=0.01):
    temperature = tf.Variable(1.0, dtype=tf.float32, trainable=True)
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    y_cal_tensor = tf.convert_to_tensor(y_cal, dtype=tf.int32)

    for _ in range(steps):
        with tf.GradientTape() as tape:
            logits = model(x_cal, training=False) / temperature
            loss = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(
                    y_cal_tensor,
                    logits,
                    from_logits=True
                )
            )
        grads = tape.gradient(loss, [temperature])
        optimizer.apply_gradients(zip(grads, [temperature]))
        temperature.assign(tf.clip_by_value(temperature, 0.5, 5.0))

    return float(temperature.numpy())


def tune_reject_thresholds(probs, y_true, min_coverage=0.85):
    pred = np.argmax(probs, axis=1)
    conf = np.max(probs, axis=1)
    top2 = np.partition(probs, -2, axis=1)[:, -2]
    margin = conf - top2

    best = None
    for conf_thr in np.linspace(0.55, 0.95, 9):
        for margin_thr in np.linspace(0.05, 0.30, 11):
            accept = (conf >= conf_thr) & (margin >= margin_thr)
            coverage = np.mean(accept)
            if coverage < min_coverage:
                continue

            acc = np.mean(pred[accept] == y_true[accept]) if np.any(accept) else 0.0
            score = acc + 0.05 * coverage

            if best is None or score > best['score']:
                best = {
                    'conf_threshold': float(conf_thr),
                    'margin_threshold': float(margin_thr),
                    'accepted_acc': float(acc),
                    'coverage': float(coverage),
                    'score': float(score)
                }

    if best is None:
        best = {
            'conf_threshold': 0.75,
            'margin_threshold': 0.15,
            'accepted_acc': 0.0,
            'coverage': 0.0,
            'score': 0.0
        }

    return best


temperature = learn_temperature(kan_model, X_cal, y_cal)
print(f'Learned temperature: {temperature:.4f}')


# Tune reject thresholds on calibration split

cal_logits = kan_model.predict(X_cal, batch_size=256)
cal_probs = tf.nn.softmax(cal_logits / temperature, axis=1).numpy()

best_thresholds = tune_reject_thresholds(cal_probs, y_cal, min_coverage=0.85)
conf_threshold = best_thresholds['conf_threshold']
margin_threshold = best_thresholds['margin_threshold']

print('Selected thresholds from calibration:')
print(f" - confidence threshold: {conf_threshold:.2f}")
print(f" - margin threshold: {margin_threshold:.2f}")
print(f" - calibration accepted accuracy: {best_thresholds['accepted_acc']:.4f}")
print(f" - calibration coverage: {best_thresholds['coverage']:.4f}")


# Evaluate with calibrated probabilities + tuned reject option

test_logits = kan_model.predict(test_features, batch_size=256)
test_probs = tf.nn.softmax(test_logits / temperature, axis=1).numpy()

pred = np.argmax(test_probs, axis=1)
conf = np.max(test_probs, axis=1)
top2 = np.partition(test_probs, -2, axis=1)[:, -2]
margin = conf - top2

reject_mask = (conf < conf_threshold) | (margin < margin_threshold)
accept_mask = ~reject_mask

overall_acc = np.mean(pred == y_test_small)
accepted_acc = np.mean(pred[accept_mask] == y_test_small[accept_mask]) if np.any(accept_mask) else 0.0
coverage = np.mean(accept_mask)

print(f'\nCalibrated overall accuracy: {overall_acc:.4f}')
print(f'Accepted-sample accuracy: {accepted_acc:.4f}')
print(f'Coverage (not rejected): {coverage:.4f}')
print(f'Rejected fraction: {np.mean(reject_mask):.4f}')

CNN model built successfully.
Epoch 1/5
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 87ms/step - accuracy: 0.8224 - loss: 0.6006 - val_accuracy: 0.1247 - val_loss: 2.1307
Epoch 2/5
106/106 ━━━━━━━━━━━━━━━━━━━━ 9s 86ms/step - accuracy: 0.9560 - loss: 0.1494 - val_accuracy: 0.2100 - val_loss: 1.9437
Epoch 3/5
106/106 ━━━━━━━━━━━━━━━━━━━━ 9s 84ms/step - accuracy: 0.9718 - loss: 0.0941 - val_accuracy: 0.7100 - val_loss: 0.9722
Epoch 4/5
106/106 ━━━━━━━━━━━━━━━━━━━━ 9s 83ms/step - accuracy: 0.9782 - loss: 0.0697 - val_accuracy: 0.9153 - val_loss: 0.3434
Epoch 5/5
106/106 ━━━━━━━━━━━━━━━━━━━━ 9s 84ms/step - accuracy: 0.9804 - loss: 0.0602 - val_accuracy: 0.9640 - val_loss: 0.1291
Feature extractor created successfully!


32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Features extracted: Train=(8000, 128), Test=(2000, 128)


Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ kan_layer_3 (KANLayer)          │ (None, 96)             │        12,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 96)             │         9,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 96)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 10)             │           970 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,762 (88.91 KB)

 Trainable params: 22,762 (88.91 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7972 - loss: 0.7772 - val_accuracy: 0.9806 - val_loss: 0.0652 - learning_rate: 8.0000e-04
Epoch 2/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9786 - loss: 0.0828 - val_accuracy: 0.9856 - val_loss: 0.0406 - learning_rate: 8.0000e-04
Epoch 3/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9842 - loss: 0.0605 - val_accuracy: 0.9881 - val_loss: 0.0318 - learning_rate: 8.0000e-04
Epoch 4/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9848 - loss: 0.0528 - val_accuracy: 0.9881 - val_loss: 0.0272 - learning_rate: 8.0000e-04
Epoch 5/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9883 - loss: 0.0429 - val_accuracy: 0.9906 - val_loss: 0.0250 - learning_rate: 8.0000e-04
Epoch 6/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9883 - loss: 0.0386 - val_accuracy: 0.9894 - val_loss: 0.0246 - learning_rate: 8.0000e-04
Epoch 7/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9908 - loss:

In [8]:
import os
import numpy as np


# Save Trained Models + calibration config

os.makedirs('models', exist_ok=True)

# Save in native Keras format (.keras)
cnn_model.save('models/cnn_model.keras')
feature_extractor.save('models/feature_extractor.keras')
kan_model.save('models/kan_model.keras')

# Save calibration/rejection settings for inference-time confidence handling
np.savez(
    'models/kan_calibration.npz',
    temperature=np.array([temperature], dtype=np.float32),
    conf_threshold=np.array([conf_threshold], dtype=np.float32),
    margin_threshold=np.array([margin_threshold], dtype=np.float32)
)

print("\nModels and calibration config saved successfully in 'models/' folder!")
print('Files created:')
print(' - models/cnn_model.keras')
print(' - models/feature_extractor.keras')
print(' - models/kan_model.keras')
print(' - models/kan_calibration.npz')


Models and calibration config saved successfully in 'models/' folder!
Files created:
 - models/cnn_model.keras
 - models/feature_extractor.keras
 - models/kan_model.keras
 - models/kan_calibration.npz
